# B2-024 — Book 1 Bridge Diagnostic

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

This 30-minute diagnostic checks the Book 1 tools that B2-024 imports without re-teaching.
Answer both checkpoints in each section before reading its collected answers.
If a section is not fluent, follow the named remediation link before Session 1.
`B2-023-generative-models-diffusion` is a schedule-order predecessor only: no Book 2 concept is required here, and Session 1's forward-only topic merely refers back to it.
Mixed precision, checkpointing with RNG state, the percentile bootstrap, pseudo-labeling, Tikhonov regularization of a grid operator, and permutation-invariant losses are **not** assumed; Sessions 1–6 teach them.

## 1. Arrays, broadcasting, and seeds

Every computation in this unit is batched: rows are examples.
Broadcasting (`book1:broadcasting`) aligns trailing dimensions, so `(N, 1, 2) - (1, S, 2)` gives `(N, S, 2)`, which is how Session 5 computes all source-to-sensor offsets at once.
Tensors behave like `book1:numpy-arrays` with a device and an autograd flag.
`book1:random-seeding` makes runs repeatable; this unit additionally restores generator *state* to resume runs exactly (Session 1) and uses dedicated seeded `torch.Generator` objects for bootstraps (Session 3).
Remediation: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb).

**Checkpoint 1A.** What is the shape of `a[:, None, :] - b[None, :, :]` for `a` of shape `(200, 2)` and `b` of shape `(8, 2)`?

**Checkpoint 1B.** Two runs seed `torch.manual_seed(0)` but one draws an extra random number before training starts. Do their mini-batches match?

**Collected answers.** `(200, 8, 2)`. No: the extra draw advances the global generator, so every later draw (initialization, batch indices) differs.

## 2. Matrices, rank, and linear least squares

`A @ u` with `A` of shape `(8, 81)` and `u` of shape `(81,)` gives `(8,)` (`book1:matrix-multiplication`).
A square matrix is invertible exactly when it has full rank (`book1:invertibility-via-rank`); `rank(AᵀA) = rank(A)`.
Linear regression (`book1:linear-regression`) minimizes the squared error (`book1:mse-loss`) and solves the normal equations; ridge regression (`book1:l2-regularization`) adds $\lambda\lVert w\rVert^2$, which Session 5 reuses as Tikhonov regularization.
Remediation: [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb) for rank and products; [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb) for least squares and ridge.

**Checkpoint 2A.** $A$ is $8\times81$ with rank 8. What is the rank of $A^\top A$, and is it invertible?

**Checkpoint 2B.** Write the ridge-regression objective for design matrix $X$, targets $y$, and coefficients $w$.

**Collected answers.** Rank 8, so the $81\times81$ matrix $A^\top A$ is singular (not invertible). $\lVert Xw-y\rVert^2+\lambda\lVert w\rVert^2$ with $\lambda>0$.

## 3. Gradients and gradient descent

The gradient (`book1:gradient`) collects partial derivatives; for a sum of squares, $\nabla_w\lVert Xw-y\rVert^2=2X^\top(Xw-y)$ (`book1:sum-of-squares-gradients`).
Gradient descent (`book1:gradient-descent`) steps against the gradient with a learning rate (`book1:learning-rate`); stochastic gradient descent (`book1:stochastic-gd`) uses a random mini-batch per step, which is why Session 1's checkpoints must restore the generator that draws the batches.
Remediation: [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb) for gradients; [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb) for descent, learning rates, and SGD.

**Checkpoint 3A.** Give $\nabla_u\big(\lVert Au-y\rVert^2+\lambda\lVert u\rVert^2\big)$.

**Checkpoint 3B.** What changes between full-batch gradient descent and SGD, and what randomness does SGD introduce?

**Collected answers.** $2A^\top(Au-y)+2\lambda u$. SGD estimates the gradient from a random mini-batch each step instead of all rows, so the sequence of sampled batches (drawn from a random generator) becomes part of the run.

## 4. Expectation, variance, covariance, and the Gaussian

`book1:expectation` is linear; `book1:variance` is $E[(X-EX)^2]$; for independent variables (`book1:independence`) the variance of a sum is the sum of variances (`book1:variance-of-sums`), and in general covariance terms (`book1:covariance`) are added.
`book1:gaussian-distribution` describes the sensor noise and the shape of each mixture bump; `book1:sampling-simulation` is how all of this unit's data are produced, from seeded generators.
Session 3 uses these facts to derive $\operatorname{Var}(\bar X)=\sigma^2/n$ and the variance of a paired difference.
Remediation: [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb).

**Checkpoint 4A.** $X$ and $Y$ are independent with variances 4 and 9. What is $\operatorname{Var}(X-Y)$?

**Checkpoint 4B.** If $\operatorname{Cov}(X,Y)=3$ instead, what is $\operatorname{Var}(X-Y)$?

**Collected answers.** $4+9=13$ (independence removes the covariance term, and the minus sign does not change a variance). $4+9-2\cdot3=7$.

## 5. Protocols, metrics, and competition craft

`book1:train-test-split` and `book1:hidden-test-protocol` separate fitting, selection, and final scoring; `book1:overfitting` is what a held-out set detects.
`book1:accuracy-precision-recall` and `book1:class-imbalance` matter when pseudo-labels skew the classes (Session 4).
Book 1's competition craft taught `book1:metric-driven-iteration` (change one thing, measure on validation, decide), `book1:writeup-quality` (a reader can reproduce your choices), `book1:colab-coding-submission` (the hosted-notebook submission routine), and `book1:cpu-and-gpu-round-boundary` (Round 1 is CPU-only; Round 2 may grant an accelerator).
Sessions 1–3 extend each of these to budgeted, GPU-aware, open-ended work.
Remediation: [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb) for splits, overfitting, and metrics; [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb) for the hidden-test protocol, iteration, writeups, Colab submission, and the round boundary.

**Checkpoint 5A.** Which split chooses hyperparameters, and how many times is the locked test set scored?

**Checkpoint 5B.** On a pool that is 70% class 0, a classifier predicts class 0 for everything. Give its accuracy and its recall on class 1.

**Collected answers.** The validation split; the test set is scored once, after all fitting and selection. Accuracy $0.7$; recall on class 1 is $0$.

## 6. Neural networks, CNNs, and the PyTorch training loop

An MLP (`book1:mlp-architecture`) alternates linear layers with `book1:relu-activation`; a classifier ends in logits, and `book1:softmax` with `book1:cross-entropy-loss` (`F.cross_entropy`) trains it.
`book1:torch-tensors`, `book1:nn-module`, and `book1:requires-grad` are the PyTorch building blocks; `book1:parameter-counting` is the basis of Session 1's memory arithmetic.
`book1:convolution` and `book1:cnn-training` give the small CNN used for the shapes tasks.
`book1:torch-optimizers` (Adam), `book1:autograd-training` (zero_grad → forward → loss → backward → step), and `book1:trained-mlp` (a trained network evaluated on held-out data) are used in every training practice.
Remediation: [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb); [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb); [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb); [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

**Checkpoint 6A.** How many parameters does `nn.Conv2d(1, 8, 3)` have, and what spatial size does it produce from an $8\times8$ input with no padding?

**Checkpoint 6B.** What goes wrong if `optimizer.zero_grad()` is omitted from the loop?

**Collected answers.** $8\cdot(1\cdot3\cdot3)+8=80$; output $6\times6$ per channel. Gradients accumulate across steps, so each update uses the sum of all past gradients.

## 7. k-means and Lloyd's algorithm

`book1:k-means` partitions points into $k$ clusters by minimizing the within-cluster sum of squared distances; `book1:lloyd-algorithm` alternates assigning each point to its nearest centroid and moving each centroid to the mean of its points.
Session 4 runs it on frozen CNN features, with a stated initialization, iteration count, and tie rules, to name clusters with a few labels.
Remediation: [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb).

**Checkpoint 7A.** Points $0, 1, 9, 10$ on a line, centroids initialized at $0$ and $10$. Give the centroids after one Lloyd iteration.

**Checkpoint 7B.** What should happen to a centroid with no assigned points, under this unit's rule?

**Collected answers.** Assignments $\{0,1\}$ and $\{9,10\}$; centroids $0.5$ and $9.5$. It stays where it is (this unit's stated rule).

## 8. Readiness and explicit remediation

Proceed to Session 1 only if every checkpoint above was fluent; otherwise use the link for the exact gap.
The unit's own new tools are taught from scratch in Sessions 1–6.
[B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb) precedes this unit in the schedule only.
The cell below checks that the unit's data module loads and that the CPU path works.

**Checkpoint 8A.** Which link do you follow if you cannot derive the gradient of a sum of squares?

**Checkpoint 8B.** Do you need a GPU to complete any required part of this unit?

**Collected answers.** [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb) (then C2 for least squares). No: every required lesson and practice runs on a CPU; accelerator extensions are optional.

In [ ]:
import importlib.util
from pathlib import Path
import torch

candidates = (Path("../data/capstone_data.py"), Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
              Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"))
path = next(p for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location("capstone_data", path)
capstone_data = importlib.util.module_from_spec(spec)
spec.loader.exec_module(capstone_data)

readiness = {
    "broadcast shape": (torch.zeros(200, 1, 2) - torch.zeros(1, 8, 2)).shape == (200, 8, 2),
    "rank of A^T A": torch.linalg.matrix_rank(torch.ones(8, 81) + torch.eye(8, 81)).item() <= 8,
    "data module tasks": capstone_data.TASKS == ("shapes_supervised", "shapes_ssl", "inverse", "mixture"),
    "split sizes": [len(capstone_data.TRAIN_IDS[t]) for t in capstone_data.TASKS] == [600, 30, 800, 800],
    "CPU path": torch.device("cuda" if torch.cuda.is_available() else "cpu").type in ("cpu", "cuda"),
}
print(readiness)
print("ready for B2-024" if all(readiness.values()) else "follow the remediation links first")